# Neural Networks and Deep Learning — Seminar 11: Explainable AI: SHAP, LIME, gradient methods and the stability of explanations
Prof. dr. Daniel Traian Pele — Bucharest University of Economic Studies (ASE); Romanian Academy, Institute for Economic Forecasting (IPE)

[Open in Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L11_seminar.ipynb)

Seminar tasks on real data from `../data/clean` (see `../data/SOURCES.md`). Write your code in the empty cells under each task; keep the seed fixed so that your numbers can be compared.

In [ ]:
# Google Colab: copy the course repository, so that the data and the saved results are found
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Setup: German credit (Statlog), 1,000 applicants

In [ ]:
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import shap
from lime.lime_tabular import LimeTabularExplainer
torch.set_num_threads(4)
class MLP(nn.Module):
    def __init__(self, d, h1=64, h2=32, p=0.1):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(d, h1), nn.ReLU(), nn.Dropout(p), nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(p), nn.Linear(h2, 1))
    def forward(self, x): return self.f(x).squeeze(-1)
def train_mlp(seed, Xa, ya, Xv, yv, w_pos=1.0, epochs=300, patience=25, task='clf', lr=1e-3):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); net = MLP(Xa.shape[1])
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-4)
    Xt, yt = torch.from_numpy(Xa), torch.from_numpy(ya.astype(np.float32)); best, state, bad = -np.inf, None, 0
    for ep in range(epochs):
        net.train(); perm = rng.permutation(len(Xa))
        for j in range(0, len(Xa), 256):
            b = torch.from_numpy(perm[j:j + 256]); out = net(Xt[b])
            if task == 'clf':
                w = torch.where(yt[b] > 0, torch.tensor(w_pos), torch.tensor(1.0))
                loss = nn.functional.binary_cross_entropy_with_logits(out, yt[b], weight=w)
            else:
                loss = ((out - yt[b]) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): pv = net(torch.from_numpy(Xv)).numpy()
        sc = (roc_auc_score(yv, pv) if w_pos == 1.0 else average_precision_score(yv, pv)) if task == 'clf' else -np.mean((pv - yv) ** 2)
        if sc > best: best, state, bad = sc, {k: v.clone() for k, v in net.state_dict().items()}, 0
        else:
            bad += 1
            if bad > patience: break
    net.load_state_dict(state); net.eval(); return net
def logit_fn(net):
    def f(X):
        with torch.no_grad(): return net(torch.from_numpy(np.asarray(X, dtype=np.float32))).numpy().astype(np.float64)
    return f
sig = lambda z: 1 / (1 + np.exp(-z))

In [ ]:
g = pd.read_csv(DATA + 'german_credit.csv'); y = g['default'].values
X = pd.get_dummies(g.drop(columns='default'), drop_first=True).astype(float); FEATS = list(X.columns)
idx = np.arange(len(g)); i_tr, i_te = train_test_split(idx, test_size=0.25, stratify=y, random_state=SEED)
i_tr, i_va = train_test_split(i_tr, test_size=0.2, stratify=y[i_tr], random_state=SEED)
mu, sd = X.iloc[i_tr].mean(), X.iloc[i_tr].std().replace(0, 1)
XS = ((X - mu) / sd).values.astype(np.float32)
Xtr, Xva, Xte, ytr, yva, yte = XS[i_tr], XS[i_va], XS[i_te], y[i_tr], y[i_va], y[i_te]
print(len(g), 'applicants,', len(FEATS), 'features')

**Task 1.** Train the MLP (seed 20261108) and a logistic regression; report the test AUC and PR-AUC of both.

In [ ]:
# your code here

**Task 2.** Compute Kernel SHAP values of the MLP for 100 test applicants (100 background applicants, 400 samples). Check additivity and list the 8 most important features.

In [ ]:
# your code here

**Task 3.** Fairness check: how large are the SHAP contributions of age_years and of the personal_status_sex dummies compared with the top feature? What would you tell a regulator?

In [ ]:
# your code here

**Task 4.** Explain the riskiest test applicant with LIME 20 times (500 samples, seeds 0-19). How often is the top feature the same?

In [ ]:
# your code here

**Task 5.** Integrated gradients for the same applicant (baseline = mean good applicant of the training set, 128 steps). Verify completeness.

In [ ]:
# your code here

**Task 6.** Rashomon check: retrain the MLP with three more seeds. Are the test AUCs close? Do the top-5 SHAP features stay the same (Kendall tau of the importances)?

In [ ]:
# your code here